In [82]:
import oracledb
import pandas as pd
import torch
import random
import json
from datetime import datetime

oracledb.init_oracle_client(lib_dir=r"D:\\instantclient_23_9")

conn = oracledb.connect(
    user="adsql",          # 사용자명
    password="oracle_4U",      # 비밀번호
    dsn="localhost:1521/xe" # 접속 정보 (SQL Developer와 동일)
)
cur = conn.cursor()

In [83]:
from prediction_all import *

In [84]:
mode = 'user_diy'

# 1. 만약 이게 user가 입력해서 만든 분자를 최적화하려는 거면
if mode == 'user_diy':
	df = pd.read_csv('user_generative.csv')
	col_start = 'U'
	orig_diff_col = ['UNEW_MOL_WEIGHT', 'UNEW_LOGP', 'UNEW_QED', 'UNEW_TOXIC','UNEW_PKI','UNEW_PKD']
else: # 버튼 누른거면
	df = pd.read_csv('disease_generative.csv')
	col_start = 'D'
	orig_diff_col = ['DNEW_MOL_WEIGHT', 'DNEW_LOGP', 'DNEW_QED', 'DNEW_TOXIC','DNEW_PKI','DNEW_PKD']

optim_diff_col = ['ONEW_MOL_WEIGHT', 'ONEW_LOGP', 'ONEW_QED', 'ONEW_TOXIC','ONEW_PKI','ONEW_PKD']

In [85]:
optim_molecule = pd.read_csv('optim_molecule.csv')
optim_molecule = optim_molecule.drop_duplicates('ONEW_CANOSMILES')

In [86]:
orig_mol = df[df[f'{col_start}NEW_NAME'].isin(optim_molecule['ONEW_ORIGIN_NAME'])]
orig_mol = orig_mol[orig_diff_col].to_numpy()
orig_mol

array([[ 4.35892000e+02,  1.50940000e+00,  2.47230792e-01,
        -3.93064270e+00,  7.60647583e+00,  4.50373936e+00]])

In [87]:
return_val_col = ['ONEW_NAME', 'ONEW_CANOSMILES', 'ONEW_IMAGE_BASE64',
 'ONEW_MOL_WEIGHT', 'ONEW_LOGP', 'ONEW_QED', 
 'ONEW_TOXIC','ONEW_PKI','ONEW_PKD']

In [103]:
return_value = []
for i in range(len(optim_molecule)):
    dic = optim_molecule[return_val_col].loc[i].to_dict()
    optim_mol = optim_molecule[optim_diff_col].loc[i].to_numpy()
    diff_res = optim_mol - orig_mol 
    dic['DIFF_INFO'] = dict(zip([i[5:] + '_DIFF' for i in optim_diff_col], diff_res[0].tolist()))
    return_value.append(dic)

In [104]:
return_value

[{'ONEW_NAME': 'ONEW_MOLECULE0',
  'ONEW_CANOSMILES': 'B[N-1][PH1]#[N+1][N-1][PH1]#C[N-1][N+1]([O-1])PN\\[NH1]C1NCCO1',
  'ONEW_IMAGE_BASE64': '...',
  'ONEW_MOL_WEIGHT': 308.9360000000001,
  'ONEW_LOGP': 0.022,
  'ONEW_QED': 0.3754743932391992,
  'ONEW_TOXIC': -4.159294128417969,
  'ONEW_PKI': 6.0259928703308105,
  'ONEW_PKD': 13.789772987365724,
  'DIFF_INFO': {'MOL_WEIGHT_DIFF': -126.95600000000002,
   'LOGP_DIFF': -1.4873999999999987,
   'QED_DIFF': 0.1282436007547386,
   'TOXIC_DIFF': -0.22865142822265616,
   'PKI_DIFF': -1.5804829597473145,
   'PKD_DIFF': 9.286033630371096}},
 {'ONEW_NAME': 'ONEW_MOLECULE1',
  'ONEW_CANOSMILES': '[N+1][N-1][PH1]#[N+1][N-1][PH1]#C[N-1][N+1]([O-1])PNC1NCCO1',
  'ONEW_IMAGE_BASE64': '...',
  'ONEW_MOL_WEIGHT': 295.10000000000014,
  'ONEW_LOGP': 0.953,
  'ONEW_QED': 0.5899431110524199,
  'ONEW_TOXIC': -4.233738327026368,
  'ONEW_PKI': 7.076668739318848,
  'ONEW_PKD': 8.532742500305176,
  'DIFF_INFO': {'MOL_WEIGHT_DIFF': -140.79199999999997,
   'LOGP_